<a href="https://colab.research.google.com/github/vhvaldiviesor/Programacion-para-analitica-descriptiva-y-predictiva/blob/main/Actividad_Calificada_Particion_Remuestreo_Arbol_269711.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Actividad Calificada: Partición y Remuestreo con Árbol de Decisión

**Programación para Analítica Descriptiva y Predictiva — MIAAD, UACJ**


**Nombre: Victor Hugo Valdivieso Rodriguez**

**Matricula: 269711**

**Ficha técnica**

| | |
|---|---|
| **Modalidad** | Individual |
| **Entorno** | Google Colab |
| **Dataset** | Titanic (pasajeros del RMS Titanic, clasificación binaria) |
| **Modelo** | Árbol de Decisión (`DecisionTreeClassifier`) |
| **Puntaje total** | 100 puntos |
| **Fecha de entrega** | 26 de octubre de 2026 |

**Objetivo**

Aplicar partición hold-out estratificada y dos esquemas de validación cruzada (`StratifiedKFold` y `RepeatedStratifiedKFold`) para estimar el desempeño de un Árbol de Decisión, y justificar, con base en el sesgo y la varianza de la estimación, qué método reportarías como resultado final.

**Dataset**

El dataset contiene 891 pasajeros del Titanic, con la variable `survived` (0 = no sobrevivió, 1 = sobrevivió) como variable objetivo. Tiene valores ausentes genuinos en `age` (177 casos) y `embarked` (2 casos) — no son valores agregados artificialmente para esta actividad.

**Instrucciones generales**

- Completa únicamente las celdas marcadas con `# TODO`. No modifiques las celdas de preparación de datos ni los hiperparámetros ya indicados (`max_depth=4`, `random_state=0`), para que tus resultados sean comparables con los de tus compañeros.
- Cada actividad tiene una parte de código y una pregunta de interpretación en una celda de texto (Markdown). Responde directamente debajo de **Tu respuesta:**.
- No se califican fragmentos de código comentados o dejados sin completar (`# TODO` sin resolver).
- Antes de entregar, ejecuta todo el notebook de principio a fin (`Entorno de ejecución > Reiniciar y ejecutar todo` en Colab) y verifica que no queden errores.

**Entrega**

Sube a Moodle un archivo `.txt` con el enlace a tu notebook de Colab o GitHub (con permisos de lectura habilitados). La subida del archivo `.ipynb` es opcional, pero recomendada como respaldo.

## Preparación de los datos

Esta sección se proporciona completa: no es parte de la evaluación. Carga el dataset, selecciona las variables predictoras y aplica un tratamiento mínimo a los valores ausentes y a las variables categóricas, para que puedas concentrarte en la partición y el remuestreo.


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.precision", 3)

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

print(f"Observaciones: {df.shape[0]}, columnas: {df.shape[1]}")
print("Valores ausentes por columna (solo las que tienen alguno):")
print(df.isna().sum()[df.isna().sum() > 0])

# Variables predictoras: se dejan fuera las que son redundantes con 'survived'
# (alive, who, adult_male), las de muy alta cardinalidad (deck, con 688 ausentes)
# o duplicadas en otra forma (class es la versión en texto de pclass).
variables = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
X = df[variables].copy()
y = df["survived"].copy()

# Tratamiento mínimo, dado: imputar age con la mediana y embarked con la moda,
# y codificar las dos variables categóricas como enteros.
X["age"] = X["age"].fillna(X["age"].median())
X["embarked"] = X["embarked"].fillna(X["embarked"].mode()[0])
X["sex"] = X["sex"].map({"male": 0, "female": 1})
X["embarked"] = X["embarked"].map({"S": 0, "C": 1, "Q": 2})

print(f"\nVariables predictoras: {list(X.columns)}")
print(f"Distribución de la variable objetivo (survived): {dict(y.value_counts().sort_index())}")


Observaciones: 891, columnas: 15
Valores ausentes por columna (solo las que tienen alguno):
age            177
embarked         2
deck           688
embark_town      2
dtype: int64

Variables predictoras: ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
Distribución de la variable objetivo (survived): {0: np.int64(549), 1: np.int64(342)}


---
## Actividad 1 — Hold-out estratificado (20 pts)

**1.1 (15 pts).** Completa la partición hold-out: reserva el 20% de las observaciones para prueba, con estratificación por `y` y `random_state=42`. Ajusta un `DecisionTreeClassifier` con `max_depth=4` y `random_state=0` usando el conjunto de entrenamiento, y calcula su accuracy sobre el conjunto de prueba.


In [2]:
# TODO 1.1: completa la partición hold-out y el ajuste del modelo.
# Revisa las condiciones indicadas arriba (porcentaje de prueba, estratificación, random_state).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

arbol = DecisionTreeClassifier(max_depth=4, random_state=0)
arbol.fit(X_train, y_train)

acc_holdout = accuracy_score(y_test, arbol.predict(X_test))

print(f"Observaciones de entrenamiento: {len(X_train)}")
print(f"Observaciones de prueba: {len(X_test)}")
print(f"Accuracy en prueba (hold-out): {acc_holdout:.4f}")

Observaciones de entrenamiento: 712
Observaciones de prueba: 179
Accuracy en prueba (hold-out): 0.7877


**1.2 (5 pts).** El Titanic tiene una proporción de supervivencia desbalanceada (aproximadamente 38% sobrevivió, 62% no). ¿Qué problema evita la estratificación (`stratify=y`) en este caso específico, y qué podría pasar con la partición si no se usara?

**Tu respuesta:**

La estratificación evita que, por azar, el conjunto de entrenamiento o el de prueba terminen con una proporción de supervivencia muy distinta a la del dataset completo (38.38% sobrevivió). Con solo 891 observaciones en total (179 en el conjunto de prueba al 20%), una partición puramente aleatoria podría, por azar, dejar por ejemplo 30% de sobrevivientes en un conjunto y 45% en el otro; mientras más pequeño el conjunto de prueba, más fácil es que esto ocurra. Esto sesgaría tanto el entrenamiento (el árbol aprendería de una proporción de clases distinta a la real) como la evaluación (el accuracy en prueba dependería de qué tan fácil o difícil resultó esa partición particular, no del desempeño real del modelo, y no sería comparable entre dos personas que corrieran el mismo experimento sin fijar la proporción). Al usar `stratify=y`, ambos conjuntos quedan con una proporción casi idéntica a la original: 38.34% en entrenamiento y 38.55% en prueba, prácticamente iguales al 38.38% del dataset completo.

---
## Actividad 2 — StratifiedKFold (25 pts)

**2.1 (15 pts).** Completa la validación cruzada con `StratifiedKFold`: 5 folds, `shuffle=True` y `random_state=42`. Usa `cross_validate` con un `DecisionTreeClassifier(max_depth=4, random_state=0)` nuevo (mismos hiperparámetros que en la Actividad 1) y reporta la media y la desviación estándar (`ddof=1`) del accuracy sobre los 5 folds.


In [3]:
# TODO 2.1: completa la validación cruzada con StratifiedKFold.
# Revisa las condiciones indicadas arriba (número de folds, shuffle, random_state).
cv_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

arbol_kfold = DecisionTreeClassifier(max_depth=4, random_state=0)
resultados_kfold = cross_validate(arbol_kfold, X, y, cv=cv_kfold, scoring="accuracy")

scores_kfold = resultados_kfold["test_score"]

print("Accuracy por fold:", np.round(scores_kfold, 4))
print(f"Media: {scores_kfold.mean():.4f}")
print(f"Desviación estándar (ddof=1): {scores_kfold.std(ddof=1):.4f}")

Accuracy por fold: [0.832 0.826 0.792 0.809 0.837]
Media: 0.8193
Desviación estándar (ddof=1): 0.0185


**2.2 (10 pts).** Compara la media del accuracy de `StratifiedKFold` con el accuracy único que obtuviste en la Actividad 1. ¿Por qué un solo valor de hold-out no es suficiente para saber si ese resultado es representativo del desempeño real del modelo?

**Tu respuesta:**

El accuracy del hold-out (78.77%) es más bajo que la media de los 5 folds de StratifiedKFold (81.93%), y de hecho cae fuera de una desviación estándar de esa media (81.93% ± 1.85%, es decir, aproximadamente entre 80.1% y 83.8%). Esto muestra justo el problema de un solo hold-out: es una única estimación que depende de qué observaciones cayeron por azar en el conjunto de prueba de esa partición en particular. Los 5 folds de StratifiedKFold van de 79.21% a 83.71%, un rango de casi 4.5 puntos porcentuales solo por el azar de qué 20% de los datos quedó fuera en cada vuelta; con un solo hold-out no hay forma de saber en qué punto de ese rango cayó el resultado obtenido, ni de medir la variabilidad asociada a la partición. Promediar sobre varias particiones, como hace k-fold, da una estimación más estable del desempeño real y, además, permite cuantificar esa variabilidad con la desviación estándar, algo que un solo número de hold-out no puede ofrecer.

---
## Actividad 3 — RepeatedStratifiedKFold (25 pts)

**3.1 (15 pts).** Completa la validación cruzada repetida: 5 folds, 5 repeticiones y `random_state=42`. Usa `cross_validate` con el mismo modelo (`DecisionTreeClassifier(max_depth=4, random_state=0)`) y reporta la media y la desviación estándar (`ddof=1`) sobre las 25 estimaciones resultantes.


In [4]:
# TODO 3.1: completa la validación cruzada repetida.
# Revisa las condiciones indicadas arriba (número de folds, número de repeticiones, random_state).
cv_repetido = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

arbol_repetido = DecisionTreeClassifier(max_depth=4, random_state=0)
resultados_repetido = cross_validate(arbol_repetido, X, y, cv=cv_repetido, scoring="accuracy")

scores_repetido = resultados_repetido["test_score"]

print(f"Número de estimaciones: {len(scores_repetido)}")
print(f"Media: {scores_repetido.mean():.4f}")
print(f"Desviación estándar (ddof=1): {scores_repetido.std(ddof=1):.4f}")

Número de estimaciones: 25
Media: 0.8088
Desviación estándar (ddof=1): 0.0217


**3.2 (10 pts).** Compara la desviación estándar de `RepeatedStratifiedKFold` con la de `StratifiedKFold` (Actividad 2). ¿Qué fuente de variabilidad adicional captura la versión repetida que una sola corrida de `StratifiedKFold` no puede capturar?

**Tu respuesta:**

La desviación estándar de RepeatedStratifiedKFold (2.17%) es mayor que la de una sola corrida de StratifiedKFold (1.85%). Esto ocurre porque, dentro de una sola corrida de 5-fold, los 5 folds son complementarios: juntos cubren el dataset completo exactamente una vez, con una sola asignación fija de qué observaciones caen en cada fold. La variabilidad que se mide ahí es solo la que existe entre esos 5 subconjuntos particulares, dada esa única forma de agruparlos. Al repetir con `n_repeats=5`, `RepeatedStratifiedKFold` vuelve a barajar las observaciones y arma una partición en 5 folds completamente distinta en cada repetición (manteniendo la estratificación). Esto captura una fuente de variabilidad adicional: cómo cambia la estimación cuando cambia la configuración completa de folds —es decir, qué observaciones quedan agrupadas juntas en cada fold—, no solo la variación entre los folds de una única partición fija. Por eso la desviación estándar de la versión repetida es más alta: refleja tanto la variabilidad dentro de una partición como la variabilidad entre distintas particiones posibles en folds.

---
## Actividad 4 — Comparación y justificación (15 pts)

**4.1 (10 pts).** Completa la tabla comparativa con los tres resultados obtenidos. Para hold-out, al ser una sola estimación, no hay desviación estándar.


In [5]:
# TODO 4.1: completa la tabla comparativa con los tres métodos.
resumen = pd.DataFrame({
    "método": ["Hold-out", "StratifiedKFold", "RepeatedStratifiedKFold"],
    "n_estimaciones": [1, len(scores_kfold), len(scores_repetido)],
    "media_accuracy": [acc_holdout, scores_kfold.mean(), scores_repetido.mean()],
    "desv_est": [np.nan, scores_kfold.std(ddof=1), scores_repetido.std(ddof=1)],
})
resumen

,método,n_estimaciones,media_accuracy,desv_est
0,Hold-out,1,0.788,NaN
1,StratifiedKFold,5,0.819,0.019
2,RepeatedStratifiedKFold,25,0.809,0.022


**4.2 (5 pts).** Si tuvieras que reportar un solo número como el desempeño esperado de este Árbol de Decisión en el dataset del Titanic, ¿cuál de los tres métodos usarías y por qué? Justifica tu respuesta en términos de sesgo y varianza de la estimación, no solo del valor de accuracy obtenido.

**Tu respuesta:**

Reportaría la media de RepeatedStratifiedKFold (80.88% ± 2.17%) como el desempeño esperado. El hold-out (78.77%) es una sola estimación: no tiene desviación estándar reportable y, como se vio en 2.2, puede caer fácilmente fuera del rango típico de las otras particiones, así que no es confiable como número único, aunque su cálculo en sí no tenga un sesgo sistemático. Entre los dos esquemas de validación cruzada, StratifiedKFold (81.93% ± 1.85%) dio el accuracy promedio más alto y la desviación estándar aparentemente más baja, pero esa desviación está subestimada: al venir de una sola partición fija en folds, solo captura la variabilidad entre esos 5 folds particulares, no la variabilidad asociada a qué folds se habrían formado con otra semilla. RepeatedStratifiedKFold, al promediar 25 estimaciones sobre 5 configuraciones distintas de folds, da una media con menor varianza de muestreo (al promediar más estimaciones independientes) y, a la vez, una desviación estándar que sí refleja honestamente toda la variabilidad por azar del proceso de partición, no solo una parte de ella. No elijo el método por cuál dio el accuracy más alto —de hecho StratifiedKFold dio el más alto—, sino porque RepeatedStratifiedKFold da la estimación menos sesgada respecto a la variabilidad real del proceso de evaluación.

---
## Reflexión final (15 pts)

Responde en 6 a 10 líneas: ¿qué papel juega la elección del método de partición o remuestreo cuando se evalúa un modelo de clasificación, y qué factores del problema (tamaño del dataset, balance de clases, costo de entrenamiento) tomarías en cuenta para elegir uno sobre otro en un caso distinto al de esta actividad?

**Tu respuesta:**

La elección del método de partición o remuestreo determina qué tan confiable es la estimación del desempeño que se reporta: un hold-out único es rápido pero da una sola muestra sujeta al azar de esa partición, mientras que los esquemas de validación cruzada promedian sobre varias particiones y permiten cuantificar la variabilidad de la estimación con una desviación estándar, no solo reportar un número suelto. El tamaño del dataset es el primer factor a considerar: con un dataset grande (cientos de miles de observaciones), un hold-out simple ya suele ser representativo y repetir particiones aporta poco a cambio de mucho cómputo extra; con un dataset pequeño, como el de esta actividad (891 filas, 179 en prueba), un solo hold-out es mucho más inestable y conviene usar k-fold o su versión repetida. El balance de clases es el segundo factor: con clases desbalanceadas, como el 38%/62% del Titanic, la estratificación (ya sea en hold-out o en k-fold) es prácticamente obligatoria para que cada partición conserve la proporción real de clases, sin importar qué tan grande sea el dataset. El tercer factor es el costo de entrenamiento: `RepeatedStratifiedKFold` multiplica el número de ajustes del modelo (en esta actividad, 25 en vez de 5 o 1), lo cual es razonable para un árbol de decisión que entrena en milisegundos, pero puede ser impráctico para un modelo costoso de entrenar (una red neuronal grande, por ejemplo), donde probablemente convenga un k-fold simple o incluso un hold-out bien diseñado como compromiso entre confiabilidad y tiempo de cómputo.

---
## Rúbrica

| Criterio | Excelente | Bueno | Insuficiente | No entregado / Incorrecto | Puntos |
|---|---|---|---|---|---|
| **1.1 Hold-out estratificado** | Partición y ajuste correctos; accuracy calculado e interpretado correctamente. | Partición y ajuste correctos, con un error menor (p. ej. un argumento incompleto) que no afecta el resultado final. | Partición o ajuste con errores que alteran el resultado (p. ej. sin estratificar, sin `random_state`). | Celda sin completar o con errores que impiden su ejecución. | 15 |
| **1.2 Interpretación de la estratificación** | Explica correctamente el riesgo de desbalance en la partición y por qué `stratify` lo evita. | Explica la idea general pero de forma incompleta o imprecisa. | Respuesta genérica, sin relación con el desbalance de clases del dataset. | Sin respuesta. | 5 |
| **2.1 StratifiedKFold** | Configuración correcta de `cv` y `cross_validate`; media y desviación estándar reportadas con `ddof=1`. | Correcto, con un error menor que no cambia el resultado (p. ej. `ddof` por defecto). | Configuración incompleta o métrica mal calculada. | Celda sin completar o con errores que impiden su ejecución. | 15 |
| **2.2 Comparación hold-out vs. k-fold** | Explica con claridad por qué una sola partición no es representativa y cómo k-fold promedia sobre varias particiones. | Identifica la idea central pero sin justificarla completamente. | Comparación superficial, sin referirse a la variabilidad entre particiones. | Sin respuesta. | 10 |
| **3.1 RepeatedStratifiedKFold** | Configuración correcta de `cv` y `cross_validate`; media y desviación estándar reportadas sobre las 25 estimaciones. | Correcto, con un error menor que no cambia el resultado. | Configuración incompleta (p. ej. `n_repeats` incorrecto) o métrica mal calculada. | Celda sin completar o con errores que impiden su ejecución. | 15 |
| **3.2 Fuente de variabilidad adicional** | Identifica correctamente que la repetición varía la configuración de folds entre repeticiones, no solo las observaciones dentro de un fold. | Identifica que hay más variabilidad capturada, sin precisar el mecanismo. | Respuesta genérica o incorrecta sobre qué varía entre repeticiones. | Sin respuesta. | 10 |
| **4.1 Tabla comparativa** | Tabla completa y correcta, con `np.nan` o vacío en la desviación estándar de hold-out. | Tabla completa con un valor incorrecto o mal ubicado. | Tabla incompleta o con varios valores incorrectos. | Celda sin completar. | 10 |
| **4.2 Justificación del método elegido** | Justifica la elección con el argumento de sesgo/varianza de la estimación, no solo con el valor numérico de accuracy. | Justifica la elección, pero basándose solo en qué método dio el accuracy más alto. | Elige un método sin justificación relacionada con sesgo o varianza. | Sin respuesta. | 5 |
| **Reflexión final** | Identifica con claridad los factores relevantes (tamaño del dataset, balance de clases, costo) y los conecta con el método de partición o remuestreo adecuado. | Identifica algunos factores relevantes, de forma parcial. | Reflexión genérica, sin relación específica con los factores de decisión vistos en la sesión. | Sin respuesta. | 15 |

**Nota.** La penalización por entrega tardía es del 5% por día calendario, hasta 7 días, después de lo cual la actividad recibe 0 puntos. Esta penalización se aplica sobre el puntaje ya obtenido con la rúbrica, no como parte de ella.
